# Day 019 — Exercise 4: run_eval

**Goal:** Implement `run_eval(test_cases, system_prompt, model, use_judge)` — the core eval loop that processes a list of TestCases through the model and returns a list of EvalResults. One real Ollama call is made in the checks.

In [ ]:
import re, ollama
from dataclasses import dataclass, field

def exact_match(response: str, expected: str) -> bool:
    return response.strip().lower() == expected.strip().lower()

def contains_any(response: str, keywords: list[str]) -> bool:
    resp_lower = response.lower()
    return any(kw.lower() in resp_lower for kw in keywords)

JUDGE_PROMPT = """\
You are an evaluation judge. Score the response below on a scale of 1 to 5.

Question: {question}
Expected answer: {expected}
Actual response: {response}

Rubric:
1 = Completely wrong or irrelevant
2 = Mostly wrong with minor correct elements
3 = Partially correct but with significant gaps
4 = Mostly correct with minor issues
5 = Fully correct and complete

Respond with ONLY this format:
Score: <1-5>
Rationale: <one sentence>
"""

def llm_judge(question, response, expected, model='llama3.2'):
    prompt = JUDGE_PROMPT.format(question=question, expected=expected, response=response)
    raw = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    text = raw['message']['content']
    m = re.search(r'Score:\s*([1-5])', text)
    score = int(m.group(1)) if m else 3
    r = re.search(r'Rationale:\s*(.+)', text)
    rationale = r.group(1).strip() if r else text.strip()[:200]
    return {'score': score, 'rationale': rationale}

@dataclass
class TestCase:
    question: str
    expected_keywords: list[str] = field(default_factory=list)
    expected_answer: str = ''

@dataclass
class EvalResult:
    test_case: TestCase
    response: str
    passed: bool
    matched_keywords: list[str] = field(default_factory=list)
    judge_score: int = 0
    judge_rationale: str = ''


## Your Implementation

In [ ]:
def run_eval(
    test_cases: list[TestCase],
    system_prompt: str = 'You are a helpful assistant.',
    model: str = 'llama3.2',
    use_judge: bool = False,
) -> list[EvalResult]:
    """
    Run all test cases against the model.

    For each TestCase:
      1. Send question to the model (with system_prompt)
      2. Find which expected_keywords appear in the response
      3. Set passed=True if any keyword matched (or no keywords expected)
      4. If use_judge=True and expected_answer set: call llm_judge
      5. Append EvalResult to results list

    Returns list[EvalResult] in the same order as test_cases.
    """
    # TODO: implement the eval loop
    pass


## Check Your Work

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0

    # Check 1: run_eval defined
    try:
        assert 'run_eval' in globals()
        passed += 1; print('✅ Check 1: run_eval defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: returns a list (one real Ollama call)
    try:
        tc = TestCase('What is the capital of France?', expected_keywords=['paris'])
        results = run_eval([tc])
        assert isinstance(results, list), f'expected list, got {type(results)}'
        assert len(results) == 1, f'expected 1 result, got {len(results)}'
        passed += 1; print('✅ Check 2: run_eval returns list with one EvalResult')
    except Exception as e:
        print(f'❌ Check 2: return type — {e}')

    # Check 3: result is an EvalResult
    try:
        r = results[0]
        assert isinstance(r, EvalResult), f'expected EvalResult, got {type(r)}'
        passed += 1; print('✅ Check 3: result is an EvalResult instance')
    except Exception as e:
        print(f'❌ Check 3: EvalResult type — {e}')

    # Check 4: response is a non-empty string
    try:
        assert isinstance(r.response, str) and len(r.response) > 0
        passed += 1; print('✅ Check 4: response is a non-empty string')
    except Exception as e:
        print(f'❌ Check 4: response — {e}')

    # Check 5: passed is a bool
    try:
        assert isinstance(r.passed, bool), f'expected bool, got {type(r.passed)}'
        passed += 1; print(f'✅ Check 5: passed is bool — got {r.passed}')
    except Exception as e:
        print(f'❌ Check 5: passed type — {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()


## Solution

<details>
<summary>Click to reveal</summary>

```python
def run_eval(test_cases, system_prompt='You are a helpful assistant.',
             model='llama3.2', use_judge=False):
    results = []
    for tc in test_cases:
        raw = ollama.chat(model=model, messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': tc.question},
        ])
        response = raw['message']['content']
        matched = [kw for kw in tc.expected_keywords
                   if kw.lower() in response.lower()]
        passed = bool(matched) if tc.expected_keywords else True
        result = EvalResult(tc, response, passed, matched)
        if use_judge and tc.expected_answer:
            j = llm_judge(tc.question, response, tc.expected_answer, model)
            result.judge_score = j['score']
            result.judge_rationale = j['rationale']
        results.append(result)
    return results
```

</details>